# PR / issue search on a GPU: dense + keyword vs keyword only

Question: given a GitHub issue, does search find the pull request that closed it? Pairs come
from GitHub's own closing links (45 closed issues of encode/httpx whose closing PR is in the
500 newest PRs), the query is the issue title plus the first 1,000 characters of its body, and
the corpus is those 500 PRs. A keyword-only (BM25) run on the laptop gave Hit@5 0.600 (95%
interval 0.467 to 0.733). This notebook adds the **hybrid** (Qwen3-Embedding-0.6B dense +
BM25, weighted RRF) number, which needs a GPU: the laptop CPU embeds about 0.1 documents/s.

**Steps:** Runtime → Change runtime type → an **A100** (or L4) GPU → Run all. The last strategy
loads the 8B reranker (Qwen3-Reranker-8B), which does not fit a 16 GB T4; to use a T4 drop
`hybrid_fused_rerank` from STRATEGIES below.
The last cell downloads `pr_search_results.zip` (report, summary) and `pr_index.zip`.

**What it embeds:** 500 PRs + 500 issues of the redacted snapshot in the notebook (emails
removed). No GitHub token or GitHub access. Not yet run on Colab: if a step fails, send the
output.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!command -v uv >/dev/null || pip install -q uv
# Pinned to the versions in the project's uv.lock. Colab already ships a CUDA build of torch.
!uv pip install --system -q "chromadb==1.5.9" "fastapi==0.162.0" "gitpython==3.2.1" "langchain-chroma==1.1.0" "langchain-core==1.6.13" "langchain-text-splitters==1.1.3" "langgraph==1.2.19" "pydantic==2.14.1" "pydantic-settings==2.16.1" "rank-bm25==0.2.2" "rich==15.0.0" "sentence-transformers==5.7.0" "transformers==4.57.6" "tree-sitter==0.26.0" "tree-sitter-language-pack==1.22.1" "typer==0.27.3" "uvicorn==0.54.0"

In [ ]:
# The repo_rag package, embedded by notebooks/build_colab_indexer.py (content hash 4338378d2005).
# Don't edit this cell: change src/repo_rag and regenerate the notebook.
import base64, io, pathlib, zipfile

ARCHIVE_B64 = "..."

SRC = pathlib.Path("/content/repo_rag_src")
with zipfile.ZipFile(io.BytesIO(base64.b64decode(ARCHIVE_B64))) as archive:
    archive.extractall(SRC)
print(f"unpacked {len(list(SRC.rglob('*.py')))} files into {SRC} (content hash 4338378d2005)")

In [ ]:
# The redacted GitHub snapshot and the eval script, embedded by notebooks/build_pr_search_eval.py
# (content hash e630d15c7c23). Don't edit this cell: regenerate the notebook instead.
import base64, io, pathlib, tarfile

PAYLOAD_B64 = "..."

DATA = pathlib.Path("/content/data")
(DATA / "github_cache" / "sync").mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD_B64)), mode="r:gz") as tar:
    for member in tar.getmembers():
        root = DATA if member.name.startswith("github_cache/") else pathlib.Path("/content")
        target = root / member.name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(tar.extractfile(member).read())
print(sorted(str(p.relative_to("/content")) for p in pathlib.Path("/content").rglob("*.json")))

In [ ]:
# Optional: a Hugging Face token makes the model download faster and avoids rate limits.
import os

try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
    print("HF_TOKEN:", "set" if token else "not set")
except Exception as exc:  # no secret, or not running in Colab
    print("HF_TOKEN: not set (", type(exc).__name__, ") - continuing without it")

## Parameters

In [ ]:
# --- Parameters -------------------------------------------------------------
DTYPE = "float32"      # matches CPU laptop queries; "bfloat16" is faster on a GPU
STRATEGIES = "bm25,hybrid,hybrid_fused,hybrid_fused_rerank"

## Embed the PRs and issues

In [ ]:
import os, subprocess, sys, time

ENV = {
    **os.environ,
    "PYTHONPATH": str(SRC),
    "REPO_RAG_DATA_DIR": "/content/data",
    "REPO_RAG_EMBEDDING_DTYPE": DTYPE,
    "PYTHONUNBUFFERED": "1",
    "TOKENIZERS_PARALLELISM": "false",
}
SLUG = "encode/httpx"


def repo_rag(*args):
    subprocess.run([sys.executable, "-m", "repo_rag.cli", *args], env=ENV, check=True)


# Load the snapshot into metadata.db exactly as `sync-github` would, without the network.
subprocess.run(
    [
        sys.executable,
        "-c",
        "from pathlib import Path\n"
        "from repo_rag.config import Settings\n"
        "from repo_rag.history.github_sync import GitHubSync\n"
        "from repo_rag.history.metadata_db import MetadataDB\n"
        "s = Settings(data_dir=Path('/content/data'))\n"
        "g = GitHubSync(s, None)\n"
        "with MetadataDB(s.metadata_db_path, readonly=False) as db:\n"
        f"    db.sync_github('{SLUG}', g.pull_requests('{SLUG}'), g.issues('{SLUG}'),"
        " prs_complete=False, issues_complete=False)\n"
        "print('metadata.db loaded')",
    ],
    env=ENV,
    check=True,
)
started = time.perf_counter()
repo_rag("index-prs", SLUG)
print(f"(index-prs took {time.perf_counter() - started:.0f} s)")

In [ ]:
repo_rag(
    "search", "retry on connection timeout", "--repo", SLUG, "--kind", "pull_requests", "-k", "3"
)

## Measure

In [ ]:
subprocess.run(
    [sys.executable, "/content/run_pr_search_eval.py", SLUG, "--data-dir", "/content/data",
     "--strategies", STRATEGIES, "--out", "/content/pr_search_results"],
    env=ENV, check=True,
)

## Download

In [ ]:
import shutil
shutil.make_archive("/content/pr_search_results", "zip", "/content/pr_search_results")
repo_rag("export-index", "/content/pr_index.zip")
from google.colab import files

files.download("/content/pr_search_results.zip")
files.download("/content/pr_index.zip")

## Afterwards

Send `pr_search_results.zip`'s `summary.md` back (or unzip it into
`eval/facts_results/httpx_pr_search_gpu/`). `pr_index.zip` can be merged locally with
`uv run repo-rag import-index pr_index.zip` so `search --kind pull_requests` uses the real
embeddings on the laptop.